In [1]:
import time
import json
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Configuration
YEAR = 2025
FILE_NAME = "match_results.json"

class ExtractIPLMatches():
    def get_matches(self):
        # Configure Chrome options
        options = webdriver.ChromeOptions()
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        
        # Update this path to your chromedriver location
        service = Service('/path/to/chromedriver')
        driver = webdriver.Chrome(options=options, service=Service(ChromeDriverManager().install()))
        
        driver.get("https://www.iplt20.com/matches/results")
        
        # Wait for page to load completely
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.ID, "team_archive"))
        )
        
        # Additional wait for dynamic content
        time.sleep(2)  # Adjust based on network speed
        
        # Find the main matches container
        archive = driver.find_element(By.ID, "team_archive")
        matches = archive.find_elements(By.TAG_NAME, "li")
        return matches

# Fetch Matches HTML
matches = ExtractIPLMatches().get_matches()

In [2]:
# kill chrome driver
#pkill -f chromedriver
#pkill -f "Google Chrome"

In [3]:
class ExtractIPLResults():
    def __init__(self):
        pass
    
    def get_text(self, html):
        if html:
            return html.text.strip()
        return None

    def find_element_by_class_name(self, element, class_name):
        try:
            return self.get_text(element.find_element(By.CLASS_NAME, class_name))
        except:
            pass
        return None

    def find_element_by_tag_name(self, element, class_name):
        try:
            return element.find_elements(By.TAG_NAME, class_name)
        except:
            pass
        return None

    def get_date(self, datetime_text):
        """
            input: "MAY, FRI 23 , 7:30 pm IST"
            output: "dd-mm-yyyy"
        """
        # Remove extra text: keep only "MAY 23"
        parts = datetime_text.split()
        month_str = parts[0]
        day_str = parts[2]        
        month_number = datetime.strptime(month_str.replace(",","").strip(), "%b").month
        
        # Format date
        date_obj = datetime(day=int(day_str), month=month_number, year=YEAR)
        formatted_date = date_obj.strftime("%d-%m-%Y")
        return formatted_date

        
    def get_match_details(self, match):
        match_number = match.find_element(By.CLASS_NAME, 'vn-matchOrder').text.strip()
        match_datetime = match.find_element(By.CLASS_NAME, 'vn-matchDateTime').text.strip()
        result = match.find_element(By.CLASS_NAME, 'vn-ticketTitle').text.strip()
        
        stadium = match.find_element(By.CLASS_NAME, 'vn-venueDet').text.strip().replace('\n', ' ')
        stadium_name_and_city = stadium.split(',')
        stadium_name = stadium_name_and_city[0]
        stadium_city = stadium_name_and_city[1]
        return {
            'match_number': match_number,
            'match_datetime': match_datetime,
            'match_date': self.get_date(match_datetime),
            'result': result,
            'stadium_name': stadium_name.strip(),
            'stadium_city': stadium_city.strip(),
        }

    def get_team1_details(self, match):
        team1_element = match.find_elements(By.CLASS_NAME, 'vn-teamTitle')[0]
        team1_name = self.find_element_by_class_name(team1_element, 'vn-teamName')
        team1_title = self.find_element_by_class_name(team1_element, 'vn-teamCode')
        team1_score_over = self.find_element_by_class_name(team1_element, 'ov-display')
        
        team1_score_p_tags = self.find_element_by_tag_name(team1_element, 'p')
        score_elements = [el for el in team1_score_p_tags if 'ng-binding' in el.get_attribute('class')]
        team1_score = None
        if len(score_elements) == 1:
            team1_score = self.get_text(score_elements[0])
            
        return {
            'team1_name': team1_name,
            'team1_title': team1_title,
            'team1_score': team1_score,
            'team1_score_over': team1_score_over
        }


    def get_team2_details(self, match):
        team2_element = match.find_elements(By.CLASS_NAME, 'vn-teamTitle')[1]
        team2_name = self.find_element_by_class_name(team2_element, 'vn-teamName')
        team2_title = self.find_element_by_class_name(team2_element, 'vn-teamCode')
        team2_score_over = self.find_element_by_class_name(team2_element, 'ov-display')
        

        team2_score_p_tags = self.find_element_by_tag_name(team2_element, 'p')
        score_elements = [el for el in team2_score_p_tags if 'ng-binding' in el.get_attribute('class')]
        team2_score = None
        if len(score_elements) == 1:
            team2_score = self.get_text(score_elements[0])

        return {
            'team2_name': team2_name,
            'team2_title': team2_title,
            'team2_score': team2_score,
            'team2_score_over': team2_score_over
        }
        
    def perform_tasks(self, matches):
        results = []
        for match in matches:
            result = {}
            result.update(self.get_match_details(match))
            result.update(self.get_team1_details(match))
            result.update(self.get_team2_details(match))
            results.append(result)            
        return results
            




In [7]:
results = ExtractIPLResults().perform_tasks(matches)

# Write results to a file
with open(FILE_NAME, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4)
    
print(json.dumps(results, indent = 3))

[
   {
      "match_number": "MATCH 65",
      "match_datetime": "MAY, FRI 23 , 7:30 pm IST",
      "match_date": "23-05-2025",
      "result": "SUNRISERS HYDERABAD WON BY 42 RUNS",
      "stadium_name": "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium",
      "stadium_city": "Lucknow",
      "team1_name": "Royal Challengers Bengaluru",
      "team1_title": "",
      "team1_score": "189",
      "team1_score_over": "(19.5 OV)",
      "team2_name": "Sunrisers Hyderabad",
      "team2_title": "",
      "team2_score": "231/6",
      "team2_score_over": "(20.0 OV )"
   },
   {
      "match_number": "MATCH 64",
      "match_datetime": "MAY, THU 22 , 7:30 pm IST",
      "match_date": "22-05-2025",
      "result": "LUCKNOW SUPER GIANTS WON BY 33 RUNS",
      "stadium_name": "Narendra Modi Stadium",
      "stadium_city": "Ahmedabad",
      "team1_name": "Gujarat Titans",
      "team1_title": "",
      "team1_score": "202/9",
      "team1_score_over": "(20.0 OV)",
      "team2_name"